# 01 — Grid Search / 01 Grid Search

**Chapter 12 — File 1 of 5 / 第12章 — 第1个文件（共5个）**

---

## Summary / 总结

This script demonstrates **grid search holt winter's exponential smoothing**.

本脚本演示 **grid search holt winter's exponential smoothing**。

---
## Step 1 — grid search holt winter's exponential smoothing

In [ ]:
from math import sqrt
from multiprocessing import cpu_count
from joblib import Parallel
from joblib import delayed
from warnings import catch_warnings
from warnings import filterwarnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
from numpy import array

---
## Step 2 — one-step Holt Winter’s Exponential Smoothing forecast

In [ ]:
def exp_smoothing_forecast(history, config):
	t,d,s,p,b,r = config

---
## Step 3 — define model

In [ ]:
history = array(history)
	model = ExponentialSmoothing(history, trend=t, damped=d, seasonal=s, seasonal_periods=p)

---
## Step 4 — fit model

In [ ]:
model_fit = model.fit(optimized=True, use_boxcox=b, remove_bias=r)

---
## Step 5 — make one step forecast

In [ ]:
yhat = model_fit.predict(len(history), len(history))
	return yhat[0]

---
## Step 6 — root mean squared error or rmse

In [ ]:
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

---
## Step 7 — split a univariate dataset into train/test sets

In [ ]:
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

---
## Step 8 — walk-forward validation for univariate data

In [ ]:
def walk_forward_validation(data, n_test, cfg):
	predictions = list()

---
## Step 9 — split dataset

In [ ]:
train, test = train_test_split(data, n_test)

---
## Step 10 — seed history with training dataset

In [ ]:
history = [x for x in train]

---
## Step 11 — step over each time-step in the test set

In [ ]:
for i in range(len(test)):

---
## Step 12 — fit model and make forecast for history

In [ ]:
yhat = exp_smoothing_forecast(history, cfg)

---
## Step 13 — store forecast in list of predictions

In [ ]:
predictions.append(yhat)

---
## Step 14 — add actual observation to history for the next loop

In [ ]:
history.append(test[i])

---
## Step 15 — estimate prediction error

In [ ]:
error = measure_rmse(test, predictions)
	return error

---
## Step 16 — score a model, return None on failure

In [ ]:
def score_model(data, n_test, cfg, debug=False):
	result = None

---
## Step 17 — convert config to a key

In [ ]:
key = str(cfg)

---
## Step 18 — show all warnings and fail on exception if debugging

In [ ]:
if debug:
		result = walk_forward_validation(data, n_test, cfg)
	else:

---
## Step 19 — one failure during model validation suggests an unstable config

In [ ]:
try:

---
## Step 20 — never show warnings when grid searching, too noisy

In [ ]:
with catch_warnings():
				filterwarnings("ignore")
				result = walk_forward_validation(data, n_test, cfg)
		except:
			error = None

---
## Step 21 — check for an interesting result

In [ ]:
if result is not None:
		print(' > Model[%s] %.3f' % (key, result))
	return (key, result)

---
## Step 22 — grid search configs

In [ ]:
def grid_search(data, cfg_list, n_test, parallel=True):
	scores = None
	if parallel:

---
## Step 23 — execute configs in parallel

In [ ]:
executor = Parallel(n_jobs=cpu_count(), backend='multiprocessing')
		tasks = (delayed(score_model)(data, n_test, cfg) for cfg in cfg_list)
		scores = executor(tasks)
	else:
		scores = [score_model(data, n_test, cfg) for cfg in cfg_list]

---
## Step 24 — remove empty results

In [ ]:
scores = [r for r in scores if r[1] != None]

---
## Step 25 — sort configs by error, asc

In [ ]:
scores.sort(key=lambda tup: tup[1])
	return scores

---
## Step 26 — create a set of exponential smoothing configs to try

In [ ]:
def exp_smoothing_configs(seasonal=[None]):
	models = list()

---
## Step 27 — define config lists

In [ ]:
t_params = ['add', 'mul', None]
	d_params = [True, False]
	s_params = ['add', 'mul', None]
	p_params = seasonal
	b_params = [True, False]
	r_params = [True, False]

---
## Step 28 — create config instances

In [ ]:
for t in t_params:
		for d in d_params:
			for s in s_params:
				for p in p_params:
					for b in b_params:
						for r in r_params:
							cfg = [t,d,s,p,b,r]
							models.append(cfg)
	return models

if __name__ == '__main__':

---
## Step 29 — define dataset

In [ ]:
data = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
	print(data)

---
## Step 30 — data split

In [ ]:
n_test = 4

---
## Step 31 — model configs

In [ ]:
cfg_list = exp_smoothing_configs()

---
## Step 32 — grid search

In [ ]:
scores = grid_search(data, cfg_list, n_test)
	print('done')

---
## Step 33 — list top 3 configs

In [ ]:
for cfg, error in scores[:3]:
		print(cfg, error)

---
## Learning Notes / 学习笔记

- **概念**: grid search holt winter's exponential smoothing 是机器学习中的常用技术。  
  *grid search holt winter's exponential smoothing is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Grid Search / 01 Grid Search
# Complete Code / 完整代码
# ===============================

# grid search holt winter's exponential smoothing
from math import sqrt
from multiprocessing import cpu_count
from joblib import Parallel
from joblib import delayed
from warnings import catch_warnings
from warnings import filterwarnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
from numpy import array

# one-step Holt Winter’s Exponential Smoothing forecast
def exp_smoothing_forecast(history, config):
	t,d,s,p,b,r = config
	# define model
	history = array(history)
	model = ExponentialSmoothing(history, trend=t, damped=d, seasonal=s, seasonal_periods=p)
	# fit model
	model_fit = model.fit(optimized=True, use_boxcox=b, remove_bias=r)
	# make one step forecast
	yhat = model_fit.predict(len(history), len(history))
	return yhat[0]

# root mean squared error or rmse
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

# split a univariate dataset into train/test sets
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

# walk-forward validation for univariate data
def walk_forward_validation(data, n_test, cfg):
	predictions = list()
	# split dataset
	train, test = train_test_split(data, n_test)
	# seed history with training dataset
	history = [x for x in train]
	# step over each time-step in the test set
	for i in range(len(test)):
		# fit model and make forecast for history
		yhat = exp_smoothing_forecast(history, cfg)
		# store forecast in list of predictions
		predictions.append(yhat)
		# add actual observation to history for the next loop
		history.append(test[i])
	# estimate prediction error
	error = measure_rmse(test, predictions)
	return error

# score a model, return None on failure
def score_model(data, n_test, cfg, debug=False):
	result = None
	# convert config to a key
	key = str(cfg)
	# show all warnings and fail on exception if debugging
	if debug:
		result = walk_forward_validation(data, n_test, cfg)
	else:
		# one failure during model validation suggests an unstable config
		try:
			# never show warnings when grid searching, too noisy
			with catch_warnings():
				filterwarnings("ignore")
				result = walk_forward_validation(data, n_test, cfg)
		except:
			error = None
	# check for an interesting result
	if result is not None:
		print(' > Model[%s] %.3f' % (key, result))
	return (key, result)

# grid search configs
def grid_search(data, cfg_list, n_test, parallel=True):
	scores = None
	if parallel:
		# execute configs in parallel
		executor = Parallel(n_jobs=cpu_count(), backend='multiprocessing')
		tasks = (delayed(score_model)(data, n_test, cfg) for cfg in cfg_list)
		scores = executor(tasks)
	else:
		scores = [score_model(data, n_test, cfg) for cfg in cfg_list]
	# remove empty results
	scores = [r for r in scores if r[1] != None]
	# sort configs by error, asc
	scores.sort(key=lambda tup: tup[1])
	return scores

# create a set of exponential smoothing configs to try
def exp_smoothing_configs(seasonal=[None]):
	models = list()
	# define config lists
	t_params = ['add', 'mul', None]
	d_params = [True, False]
	s_params = ['add', 'mul', None]
	p_params = seasonal
	b_params = [True, False]
	r_params = [True, False]
	# create config instances
	for t in t_params:
		for d in d_params:
			for s in s_params:
				for p in p_params:
					for b in b_params:
						for r in r_params:
							cfg = [t,d,s,p,b,r]
							models.append(cfg)
	return models

if __name__ == '__main__':
	# define dataset
	data = [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0]
	print(data)
	# data split
	n_test = 4
	# model configs
	cfg_list = exp_smoothing_configs()
	# grid search
	scores = grid_search(data, cfg_list, n_test)
	print('done')
	# list top 3 configs
	for cfg, error in scores[:3]:
		print(cfg, error)

---

➡️ **Next / 下一步**: File 2 of 5